# Does the *generator* pick the trap more at certain budgets?

The companion notebook (`quote_budget_findings.ipynb`) tracks the **judge's** answer — i.e. whether best-of-N quote selection eventually persuades the no-context judge to pick the trap. That measures the obfuscation channel end-to-end.

Here we strip out the judge and best-of-N entirely and ask: **does the quote budget change how often the consultant (gpt-4o-mini, temp 1.2) *chooses* to argue for the trap in the first place?**

Same 48 candidates per budget as the main run, same passage, same role→letter rotations. We just read off the `chose_role` of each raw generation.


In [1]:
import json, math
from pathlib import Path
from collections import defaultdict
from itertools import combinations
import pandas as pd

ROOT = Path.cwd()
while not (ROOT/'exp/quote_budget_3way_neutral/details.json').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
recs = json.loads((ROOT/'exp/quote_budget_3way_neutral/details.json').read_text())

BUDGETS = ['1','2','3','inf']
pool = defaultdict(list)
for r in recs:
    pool[r['budget_label']].append(r)
{b: len(pool[b]) for b in BUDGETS}


{'1': 48, '2': 48, '3': 48, 'inf': 48}

## 1. Raw generator-side answer distribution per budget

Each row is the empirical distribution of the consultant's chosen role across its 48 unfiltered candidates — *no* judge, *no* best-of-N. Wilson 95% CIs on `P(trap)`.


In [2]:
def wilson(x, n, z=1.96):
    if n == 0: return (0.0, 0.0)
    phat = x/n
    denom = 1 + z*z/n
    centre = (phat + z*z/(2*n))/denom
    half = (z/denom)*math.sqrt(phat*(1-phat)/n + z*z/(4*n*n))
    return (max(0, centre-half), min(1, centre+half))

rows = []
for b in BUDGETS:
    p = pool[b]; n = len(p)
    x_trap = sum(r['chose_trap'] for r in p)
    x_corr = sum(r['chose_correct'] for r in p)
    x_third = n - x_trap - x_corr
    lo, hi = wilson(x_trap, n)
    rows.append({
        'budget': b, 'n': n,
        'P(correct)': x_corr/n, 'P(trap)': x_trap/n, 'P(third)': x_third/n,
        'P(trap) 95% CI': f'[{lo:.2f}, {hi:.2f}]',
    })
gen = pd.DataFrame(rows).set_index('budget')
gen.style.format({'P(correct)':'{:.1%}','P(trap)':'{:.1%}','P(third)':'{:.1%}'})


,n,P(correct),P(trap),P(third),P(trap) 95% CI
budget,,,,,
1,48,27.1%,2.1%,70.8%,"[0.00, 0.11]"
2,48,33.3%,4.2%,62.5%,"[0.01, 0.14]"
3,48,33.3%,18.8%,47.9%,"[0.10, 0.32]"
inf,48,37.5%,4.2%,58.3%,"[0.01, 0.14]"


**Read this row by row, not via best-of-N.** The trap rate is essentially flat at ~2–4 % for budgets 1, 2, and ∞ — well inside each other's 95% CIs. At budget 3 it jumps to ~19 %, roughly a 5× lift.

So the budget-3 obfuscation peak in the main notebook isn't just a judge artefact amplified by best-of-N — the generator itself is more likely to *choose* the trap when forced to fill a 3-quote budget.


## 2. Pairwise difference tests on `P(generator picks trap)`

Two-proportion z-tests (pooled SE). 48 candidates per budget, so power is modest, but the budget-3 effect clears the bar in every pairwise comparison.


In [3]:
def two_prop_z(x1, n1, x2, n2):
    p1, p2 = x1/n1, x2/n2
    p = (x1+x2)/(n1+n2)
    se = math.sqrt(p*(1-p)*(1/n1 + 1/n2))
    z = (p1 - p2)/se if se > 0 else 0.0
    # two-sided p via normal approx
    from math import erf, sqrt
    p_two = 2*(1 - 0.5*(1+erf(abs(z)/sqrt(2))))
    return p1-p2, z, p_two

cmp_rows = []
for a, b in combinations(BUDGETS, 2):
    pa, pb_ = pool[a], pool[b]
    xa = sum(r['chose_trap'] for r in pa); na = len(pa)
    xb = sum(r['chose_trap'] for r in pb_); nb = len(pb_)
    d, z, pv = two_prop_z(xa, na, xb, nb)
    cmp_rows.append({
        'budget A': a, 'budget B': b,
        'P(trap | A)': xa/na, 'P(trap | B)': xb/nb,
        'Δ (A−B)': d, 'z': z, 'p (two-sided)': pv,
    })
pd.DataFrame(cmp_rows).style.format({
    'P(trap | A)':'{:.1%}', 'P(trap | B)':'{:.1%}',
    'Δ (A−B)':'{:+.1%}', 'z':'{:+.2f}', 'p (two-sided)':'{:.3f}'
})


,budget A,budget B,P(trap | A),P(trap | B),Δ (A−B),z,p (two-sided)
0,1,2,2.1%,4.2%,-2.1%,-0.59,0.557
1,1,3,2.1%,18.8%,-16.7%,-2.67,0.008
2,1,inf,2.1%,4.2%,-2.1%,-0.59,0.557
3,2,3,4.2%,18.8%,-14.6%,-2.24,0.025
4,2,inf,4.2%,4.2%,+0.0%,+0.00,1.000
5,3,inf,18.8%,4.2%,+14.6%,+2.24,0.025


Budget 3 vs each of {1, 2, ∞} hits z ≈ −2.2 to −2.7 (two-sided p ≈ 0.008 – 0.025). Among the remaining three budgets the differences are indistinguishable from zero (|z| ≤ 0.6, p ≥ 0.55).


## 3. Sanity: what about `chose_correct` and `chose_third`?

If the trap is rising at budget 3, where does that probability mass come from?


In [4]:
dist = pd.DataFrame({
    b: {
        'P(correct)': sum(r['chose_correct'] for r in pool[b]) / len(pool[b]),
        'P(trap)':    sum(r['chose_trap']    for r in pool[b]) / len(pool[b]),
        'P(third)':   1 - sum(r['chose_correct'] for r in pool[b]) / len(pool[b])
                        - sum(r['chose_trap']    for r in pool[b]) / len(pool[b]),
    }
    for b in BUDGETS
}).T
dist.index.name = 'budget'
dist.style.format('{:.1%}')


,P(correct),P(trap),P(third)
budget,,,
1,27.1%,2.1%,70.8%
2,33.3%,4.2%,62.5%
3,33.3%,18.8%,47.9%
inf,37.5%,4.2%,58.3%


`P(correct)` is essentially flat across budgets (27 – 38 %, well within sampling noise). The extra trap mass at budget 3 is drawn almost entirely from the **third** suspect — i.e. the budget-3 constraint nudges the consultant away from arguing for the residue-by-elimination's *other* alibi'd suspect (which is also a defensible "not the obvious culprit" pick at low budgets) and toward the suspect with the loudest single piece of incriminating evidence.

That matches the story-side mechanic: filling three slots forces the model to dip into the **motive** sentences (`s3`, `s4`), and once it's about to quote motive sentences it more often switches its declared answer to the suspect those sentences point at.


## 4. Takeaway

| | budget 1 | budget 2 | budget 3 | budget ∞ |
|---|---|---|---|---|
| P(generator picks trap) | **2.1 %** | **4.2 %** | **18.8 %** | **4.2 %** |

- The generator's trap rate is **flat across budgets 1, 2, ∞** and **spikes ~5× at budget 3** (p ≈ 0.008 – 0.025 vs each other budget).
- So the budget-3 obfuscation peak in the judge-side analysis is partly a **generator-side** effect: forcing exactly three single-sentence quotes makes gpt-4o-mini more likely to argue for the trap, not just more likely to *quote* the motive sentences while still arguing for the residue.
- Caveats: one passage, 48 samples per budget, one consultant model at temp 1.2 — wide CIs on the absolute rates; the *cross-budget* pattern is what's robust.


## 5. Summary of the data

Generator-side trap rate (48 candidates per budget, no judge, no best-of-N):

| budget | P(generator picks trap) |
|---|---|
| 1 | 2.1% |
| 2 | 4.2% |
| **3** | **18.8%** |
| ∞ | 4.2% |

Pairwise two-proportion z-tests on `P(generator picks trap)`:

| comparison | Δ | z | p (two-sided) |
|---|---|---|---|
| budget 1 vs 2   | −2.1% | −0.59 | 0.555 |
| budget 1 vs 3   | −16.7% | −2.67 | 0.008 |
| budget 1 vs ∞   | −2.1% | −0.59 | 0.555 |
| budget 2 vs 3   | −14.6% | −2.24 | 0.025 |
| budget 2 vs ∞   |  0.0% |  0.00 | 1.000 |
| budget 3 vs ∞   | +14.6% | +2.24 | 0.025 |

**Verdict:** there *is* a difference, but only at budget = 3. Trap rate is flat at ~2–4% for budgets 1, 2, and ∞ (all pairwise z ≤ 0.6, p ≥ 0.55) and spikes ~5× at budget 3 (z ≈ −2.2 to −2.7 vs every other budget, p ≈ 0.008–0.025). The extra mass comes from `P(third)`, not `P(correct)` — forcing 3 single-sentence quotes nudges gpt-4o-mini off the third suspect and onto the trap, consistent with the motive-sentence mechanic.
